# Conservative GPS track cleaning

This notebook keeps raw `.track` files unchanged. It removes only unusable rows, never imputes longitude/latitude/timestamps, and starts a new segment after an invalid row, non-increasing timestamp, or gap greater than 10 seconds.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Student folder').exists() and (PROJECT_ROOT.parent / 'Student folder').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SOURCE_DIR = PROJECT_ROOT / 'src' / 'data'
if SOURCE_DIR.exists():
    sys.path.insert(0, str(SOURCE_DIR))
from clean_tracks import run_cleaning

DATA_ROOT = PROJECT_ROOT / 'Student folder'
OUTPUT_DIR = PROJECT_ROOT / 'cleaning_output'

## Run the cleaning pipeline

Outputs are saved in `cleaning_output/` and the DataFrames remain available in this notebook.

In [ ]:
cleaned_points, cleaning_summary, dropped_rows = run_cleaning(DATA_ROOT, OUTPUT_DIR)
print(f'Retained points: {len(cleaned_points):,}')
print(f'Dropped rows: {len(dropped_rows):,}')
print(f'Trips processed: {len(cleaning_summary):,}')
cleaning_summary

In [ ]:
# Review any rows removed by validation or exact-duplicate handling.
dropped_rows

In [ ]:
# Review the new segment boundaries used by downstream map matching.
cleaned_points.loc[cleaned_points['is_segment_start'],
                   ['partition', 'trip_id', 'original_row_index', 'segment_id', 'cleaning_reason']].head(30)

## Outputs

- `cleaned_gps_points.csv`: modelling-ready retained points with `segment_id`.
- `dropped_gps_rows.csv`: every removal and its reason.
- `cleaning_summary.csv`: one row per trip.
- `segment_summary.csv` and `segment_length_distribution.png`: segmentation audit.
- `cleaning_notes.txt`: documented cleaning policy.